# Teil 4: Evaluation des Modells

In diesem Notebook wird das Modell aus **Teil 3** ausgewertet. Ziel ist es, zu prüfen, welche Felder wichtig sind, wie gut das Modell misst und wie zuverlässig es Spieler mit vielen Toren erkennt.

In [1]:
# Benötigte Bibliotheken importieren
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix
)

import matplotlib.pyplot as plt

## 1. Datensatz laden

Die Datei **LB.xlsx** muss im gleichen Ordner wie dieses Notebook liegen.

In [2]:
# Datensatz laden
df = pd.read_excel("LB.xlsx")

# Kurze Kontrolle
display(df.head())
print("Anzahl Zeilen und Spalten:", df.shape)

,Lastname,Name,Team,Position,Games Played,Goals,Missed,Field Goals,Penalty Goals,Shooting Accuracy,Assists,Technical Faults,Steals,Blocks,Yellow Cards,2 Min Exclusion,Red Cards,Blue Cards,Season,Tier
0,Savvas,Savvas,HIL,LB,36,289,195,210,79,5971,52,40,7,7,20,16,1,0,17/18,B
1,Gunnarsson,Arnor Thor,BHC,RW,38,275,93,163,112,7473000000000001,14,17,11,0,2,3,0,0,17/18,B
2,Smits,Kay,WHV,RB,37,258,141,178,80,6466,90,74,20,4,5,10,0,0,17/18,B
3,Kaletsch,Paul,KON,LB,35,251,194,142,109,564,74,62,4,6,13,12,0,0,17/18,B
4,Billek,Florian,COB,RW,37,243,98,132,111,7126,24,29,20,0,4,2,0,0,17/18,B


Anzahl Zeilen und Spalten: (5825, 20)


## 2. Gleiche Features wie in Teil 3 verwenden

Damit die Evaluation fair ist, wird das gleiche Vorgehen wie im Notebook **model.ipynb** verwendet. Die Zielvariable ist wieder **Goals**. Die Spalten **Field Goals** und **Penalty Goals** werden nicht als Features verwendet, weil sie direkt mit der Zielvariable zusammenhängen würden.

In [3]:
# Zielvariable
target = "Goals"

# Numerische Features ohne direkte Ziel-Leakage
numeric_features = [
    "Games Played",
    "Missed",
    "Assists",
    "Technical Faults",
    "Steals",
    "Blocks",
    "Yellow Cards",
    "2 Min Exclusion",
    "Red Cards",
    "Blue Cards"
]

# Kategorische Features
categorical_features = [
    "Position",
    "Team",
    "Season",
    "Tier"
]

X = df[numeric_features + categorical_features]
y = df[target]

print("Verwendete numerische Features:")
print(numeric_features)

print("\nVerwendete kategorische Features:")
print(categorical_features)

Verwendete numerische Features:
['Games Played', 'Missed', 'Assists', 'Technical Faults', 'Steals', 'Blocks', 'Yellow Cards', '2 Min Exclusion', 'Red Cards', 'Blue Cards']

Verwendete kategorische Features:
['Position', 'Team', 'Season', 'Tier']


## 3. Train-Test-Aufteilung und Modell erneut berechnen

Das Modell wird erneut berechnet, damit dieses Notebook selbstständig funktioniert. Die Aufteilung ist gleich wie in Teil 3.

In [ ]:
# Train-Test-Aufteilung
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Vorverarbeitung und Modell
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        max_depth=12
    ))
])

# Modell trainieren
model.fit(X_train, y_train)

# Vorhersagen berechnen
y_pred = model.predict(X_test)
y_pred_rounded = np.maximum(0, np.round(y_pred)).astype(int)

print("Modell wurde trainiert.")
print("Anzahl Testdaten:", len(y_test))

## 4. Wichtige und aussagekräftige Felder bestimmen

Bei einem Random-Forest-Modell kann man die Wichtigkeit der Eingabefelder über die **Feature Importance** bestimmen. Höhere Werte bedeuten, dass ein Feld stärker zur Vorhersage beiträgt.

In [ ]:
# Feature-Namen aus der Pipeline holen
onehot = model.named_steps["preprocessor"].named_transformers_["cat"]
encoded_cat_features = list(onehot.get_feature_names_out(categorical_features))

all_feature_names = numeric_features + encoded_cat_features

# Feature Importance aus dem Random Forest holen
importances = model.named_steps["regressor"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

# Top 15 anzeigen
top_features = feature_importance.head(15)
display(top_features)

In [ ]:
# Grafik der wichtigsten Features
plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Wichtigkeit")
plt.ylabel("Feature")
plt.title("Top 15 wichtigste Features für die Vorhersage von Goals")
plt.tight_layout()
plt.show()

**Erklärung der wichtigsten Felder:**  
Die wichtigsten Felder zeigen, welche Informationen das Modell am stärksten für die Vorhersage von **Goals** nutzt. Bei diesem Datensatz sind besonders Felder wie gespielte Spiele, Fehlwürfe, Assists und weitere Leistungsstatistiken wichtig. Das ist sinnvoll, weil Spieler mit mehr Einsatzzeit und mehr offensiven Aktionen meistens auch mehr Tore erzielen.

## 5. Geeignete Messmetrik wählen und berechnen

Da **Goals** eine Zahl ist, handelt es sich um ein Regressionsproblem. Deshalb eignen sich **MAE**, **RMSE** und **R²**. Besonders gut verständlich ist der **MAE**, weil er zeigt, um wie viele Tore das Modell durchschnittlich danebenliegt.

In [ ]:
# Metriken berechnen
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metrik": ["MAE", "RMSE", "R²"],
    "Wert": [mae, rmse, r2],
    "Bedeutung": [
        "Durchschnittlicher absoluter Fehler in Toren",
        "Grösser gewichteter Fehler in Toren",
        "Erklärte Varianz des Modells"
    ]
})

display(metrics)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")

## 6. Geeignete Berechnungen

Hier werden echte und vorhergesagte Werte miteinander verglichen. Zusätzlich wird die absolute Abweichung berechnet.

In [ ]:
# Vergleichstabelle erstellen
evaluation_df = X_test.copy()
evaluation_df["Actual Goals"] = y_test.values
evaluation_df["Predicted Goals"] = y_pred_rounded
evaluation_df["Absolute Error"] = abs(evaluation_df["Actual Goals"] - evaluation_df["Predicted Goals"])

# Namen ergänzen, damit die Kontrolle besser lesbar ist
player_info = df.loc[X_test.index, ["Lastname", "Name"]]
evaluation_df = pd.concat([player_info.reset_index(drop=True), evaluation_df.reset_index(drop=True)], axis=1)

# Die 15 besten Vorhersagen anzeigen
display(evaluation_df.sort_values("Absolute Error").head(15))

# Die 15 schlechtesten Vorhersagen anzeigen
display(evaluation_df.sort_values("Absolute Error", ascending=False).head(15))

In [ ]:
# Grafik: echte Werte gegen vorhergesagte Werte
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rounded, alpha=0.5)
plt.xlabel("Echte Goals")
plt.ylabel("Vorhergesagte Goals")
plt.title("Echte Goals vs. vorhergesagte Goals")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.tight_layout()
plt.show()

## 7. Wahrheitsmatrix, Sensitivität und Spezifizität

Da das eigentliche Modell eine Zahl vorhersagt, wird für die Wahrheitsmatrix eine zusätzliche Bedingung definiert:

**Ein Spieler gilt als torgefährlich, wenn er mindestens 50 Goals erzielt hat.**

Damit wird aus der Regression eine einfache Ja/Nein-Auswertung gemacht.

In [ ]:
# Bedingung für Wahrheitsmatrix
threshold = 50

# Klassen bilden: 1 = mindestens 50 Goals, 0 = weniger als 50 Goals
actual_high_scorer = (y_test >= threshold).astype(int)
predicted_high_scorer = (y_pred >= threshold).astype(int)

# Wahrheitsmatrix berechnen
cm = confusion_matrix(actual_high_scorer, predicted_high_scorer, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

confusion_df = pd.DataFrame(
    cm,
    index=["Echt: < 50 Goals", "Echt: >= 50 Goals"],
    columns=["Vorhergesagt: < 50 Goals", "Vorhergesagt: >= 50 Goals"]
)

display(confusion_df)

print("True Negative:", tn)
print("False Positive:", fp)
print("False Negative:", fn)
print("True Positive:", tp)

In [ ]:
# Sensitivität und Spezifizität berechnen
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

classification_metrics = pd.DataFrame({
    "Metrik": ["Sensitivität", "Spezifizität", "Accuracy"],
    "Wert": [sensitivity, specificity, accuracy],
    "Bedeutung": [
        "Anteil korrekt erkannter Spieler mit mindestens 50 Goals",
        "Anteil korrekt erkannter Spieler mit weniger als 50 Goals",
        "Gesamtanteil korrekter Ja/Nein-Entscheidungen"
    ]
})

display(classification_metrics)

print(f"Sensitivität: {sensitivity:.3f}")
print(f"Spezifizität: {specificity:.3f}")
print(f"Accuracy: {accuracy:.3f}")

## 8. Zusammenfassung der Evaluation

Das Modell funktioniert insgesamt gut, weil es die Anzahl Tore mit einem relativ kleinen durchschnittlichen Fehler vorhersagt. Besonders wichtig sind Leistungsfelder wie gespielte Spiele, Fehlwürfe und Assists, was fachlich sinnvoll ist. Bei sehr hohen oder sehr niedrigen Torwerten können grössere Abweichungen entstehen, weil einzelne Spieler ungewöhnliche Rollen oder Einsatzzeiten haben. Die Wahrheitsmatrix zeigt zusätzlich, dass das Modell torgefährliche Spieler ab 50 Goals zuverlässig erkennt.

## Fazit zu Teil 4

In diesem Notebook wurden die wichtigsten Felder bestimmt, passende Messmetriken berechnet und eine Wahrheitsmatrix erstellt. Zusätzlich wurden Sensitivität und Spezifizität berechnet. Damit sind die Anforderungen für Teil 4 erfüllt.